### 03 - Camada Gold | Perguntas de negócio

## Projeto: Pipeline de Imóveis BH
## 
## AUTOR : Eduardo Cosme Pereira dos santos

## Objetivo: 

## Analises de preço e custo :
- Preço médio por região
- Preço médio por tipo de imóvel
- Ranking dos bairros mais caros
- Ranking dos bairros mais baratos
- Custo total médio por região
- Relação preço x área (valor m²)

## Análises de mercado :
- Quantidade de imóveis por bairro
- Distribuição por tipo de imóvel

 




In [0]:
## Importando bibliotecas
from pyspark.sql.functions import *
from pyspark.ml.stat import Correlation
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go




In [0]:
## configurando os logs do projeto 
import logging
logging.basicConfig(
    level=logging.INFO, filename= "/Volumes/workspace/zap_imoveis/gold/gold.log",
    format="%(asctime)s - %(levelname)s - %(message)s"
)


logging.info("logs da gold criados com sucesso!")

In [0]:
## ler os dados da Silver 

path_silver = "/Volumes/workspace/zap_imoveis/silver"

path_gold = "/Volumes/workspace/zap_imoveis/gold"

zap_gold = spark.read.format("delta").load(f"{path_silver}/zap_silver")



display(zap_gold.head(10))



logging.info("dados da silver lidos com sucesso!")




PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,BANHEIROS,VAGAS,IPTU,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO,CUSTO_TOTAL,DATA_CARGA
4500000,0,100,3,SAVASSI,3,3.0,1500,CASA,RESIDENCIAL,CENTRO-SUL,45000.0,4501500,2026-05-17T15:40:55.088Z
335000,330,100,3,BURITIS,2,1.0,145,APARTAMENTO,RESIDENCIAL,BARREIRO,3350.0,335475,2026-05-17T15:40:55.088Z
21000,0,332,3,COMITECO,6,3.0,7764,CASA,RESIDENCIAL,NOROESTE,63.25,28764,2026-05-17T15:40:55.088Z
25000,2200,264,3,LOURDES,3,3.0,1259,COBERTURA,RESIDENCIAL,CENTRO-SUL,94.7,28459,2026-05-17T15:40:55.088Z
25000,2300,105,3,SAVASSI,3,0.0,764,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,238.1,28064,2026-05-17T15:40:55.088Z
18000,4074,304,3,BELVEDERE,2,4.0,1334,COBERTURA,RESIDENCIAL,CENTRO-SUL,59.21,23408,2026-05-17T15:40:55.088Z
14000,0,256,3,BELVEDERE,4,4.0,7206,CASA,RESIDENCIAL,CENTRO-SUL,54.69,21206,2026-05-17T15:40:55.088Z
16500,2000,150,3,ANCHIETA,4,4.0,850,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,19350,2026-05-17T15:40:55.088Z
16500,2000,150,3,ANCHIETA,4,4.0,219,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,18719,2026-05-17T15:40:55.088Z
11000,0,365,3,SÃO PEDRO,2,3.0,6038,IMOVEL COMERCIAL,COMERCIAL,CENTRO-SUL,30.14,17038,2026-05-17T15:40:55.088Z


In [0]:
## Inserira coluna id para identificar os imoveis 
zap_gold = zap_gold.withColumn("ID", monotonically_increasing_id())
display(zap_gold.head(10))

PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,BANHEIROS,VAGAS,IPTU,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO,CUSTO_TOTAL,DATA_CARGA,ID
4500000,0,100,3,SAVASSI,3,3.0,1500,CASA,RESIDENCIAL,CENTRO-SUL,45000.0,4501500,2026-05-17T15:40:55.088Z,0
335000,330,100,3,BURITIS,2,1.0,145,APARTAMENTO,RESIDENCIAL,BARREIRO,3350.0,335475,2026-05-17T15:40:55.088Z,1
21000,0,332,3,COMITECO,6,3.0,7764,CASA,RESIDENCIAL,NOROESTE,63.25,28764,2026-05-17T15:40:55.088Z,2
25000,2200,264,3,LOURDES,3,3.0,1259,COBERTURA,RESIDENCIAL,CENTRO-SUL,94.7,28459,2026-05-17T15:40:55.088Z,3
25000,2300,105,3,SAVASSI,3,0.0,764,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,238.1,28064,2026-05-17T15:40:55.088Z,4
18000,4074,304,3,BELVEDERE,2,4.0,1334,COBERTURA,RESIDENCIAL,CENTRO-SUL,59.21,23408,2026-05-17T15:40:55.088Z,5
14000,0,256,3,BELVEDERE,4,4.0,7206,CASA,RESIDENCIAL,CENTRO-SUL,54.69,21206,2026-05-17T15:40:55.088Z,6
16500,2000,150,3,ANCHIETA,4,4.0,850,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,19350,2026-05-17T15:40:55.088Z,7
16500,2000,150,3,ANCHIETA,4,4.0,219,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,18719,2026-05-17T15:40:55.088Z,8
11000,0,365,3,SÃO PEDRO,2,3.0,6038,IMOVEL COMERCIAL,COMERCIAL,CENTRO-SUL,30.14,17038,2026-05-17T15:40:55.088Z,9


In [0]:
## Verificar total de linhas e colunas do dataset
print(f"Total de linhas: {zap_gold.count()}")
print(f"Total de colunas: {len(zap_gold.columns)}")

## Salvar dados na camada gold
zap_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(f"{path_gold}/zap_gold")

logging.info("dados gravado com sucesso na camada gold!")

Total de linhas: 1050
Total de colunas: 15


-  Antes das analises na camada silver ,foi verificado um valor extremo possivel outlier , vou verificar e fazer o tratamento 


In [0]:
# Buscar metrica da camada silver 
gold_metricas = spark.read.format("delta").load(f"{path_silver}/outliers")
display(gold_metricas.head(10))

## salvar os dados na camada gold
gold_metricas.write.format("delta").mode("overwrite").save(f"{path_gold}/gold_metricas")



logging.info("metricas gravado com sucesso na camada gold!")
### Salvar os dados na camada gold





PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,BANHEIROS,VAGAS,IPTU,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO,CUSTO_TOTAL
4500000,0,100,3,SAVASSI,3,3.0,1500,CASA,RESIDENCIAL,CENTRO-SUL,45000.0,4501500


##Na Durante a análise dos imóveis para aluguel, foi identificado um registro com valor significativamente acima da distribuição padrão. Após validação, há indícios de que o anúncio esteja categorizado incorretamente, possivelmente tratando-se de um imóvel para venda em vez de locação. 

## Vou excluir ele das analises da gold porem não sera retirado do dataset .

###Analises de preço e custo :
Preço médio por região

In [0]:
# Preço medio por região e gerar um grafico de barras
zap_media_regiao = zap_gold.filter(col("PRECO")<=1000000)\
    .groupBy("REGIÃO") \
    .agg(mean("PRECO").alias("PRECO_MEDIO"))\
    .withColumn("PRECO_MEDIO", round(col("PRECO_MEDIO"), 2))\
    .orderBy(col("PRECO_MEDIO").desc())
    

display(zap_media_regiao)

# Plotar um grafico de barras
fig = px.bar(zap_media_regiao, x="REGIÃO", y="PRECO_MEDIO", color="REGIÃO", title="Preço médio por região")
fig.show()
### Salvar os dados na camada gold
zap_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(f"{path_gold}/zap_media_regiao")

logging.info("media por região gravado com sucesso na camada gold!")


REGIÃO,PRECO_MEDIO
BARREIRO,5416.16
CENTRO-SUL,4904.39
OESTE,4108.02
NORTE,3868.58
NORDESTE,3672.87
NOROESTE,3514.59
OUTROS,2893.7


###Analises de preço e custo :
Preço médio por tipo de imóvel

In [0]:
# Preço medio por tipo de imovel e gerar um grafico de barras
zap_media_tipo = zap_gold.filter(col("PRECO")<=1000000).groupBy("TIPO_DE_IMOVEL") \
    .agg(mean("PRECO").alias("PRECO_MEDIO"))\
    .withColumn("PRECO_MEDIO", round(col("PRECO_MEDIO"), 2))\
    .orderBy(col("PRECO_MEDIO").desc())
    

display(zap_media_tipo)

# Plotar um grafico de barras
fig = px.bar(zap_media_tipo, x="TIPO_DE_IMOVEL", y="PRECO_MEDIO", color="TIPO_DE_IMOVEL", title="Preço médio por tipo de imovel")
fig.show()

### Salvar os dados na camada gold
zap_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(f"{path_gold}/zap_media_tipo")

logging.info("media por tipo de imovel gravado com sucesso na camada gold!")


TIPO_DE_IMOVEL,PRECO_MEDIO
CONDOMINIO,8925.0
IMOVEL COMERCIAL,6191.67
COBERTURA,5386.35
CASA,4534.84
PREDIO COMERCIAL,4500.0
APARTAMENTO,4452.3
ESCRITORIO,4114.29
EMPRESA,4033.33
null,2350.0


###Analises de preço e custo :
Ranking dos bairros mais caros

In [0]:
# Rank dos 10 bairros mais caros !
zap_bairros = zap_gold.filter(col("PRECO")<=1000000).groupBy("BAIRRO") \
    .agg(mean("PRECO").alias("PRECO_MEDIO"))\
    .withColumn("PRECO_MEDIO", round(col("PRECO_MEDIO"), 2))\
    .orderBy(col("PRECO_MEDIO").desc())\
    .limit(10)
    

display(zap_bairros)



# Plotar um grafico de 
fig = px.histogram(zap_bairros, x="BAIRRO", y="PRECO_MEDIO", color="BAIRRO", title="Preço médio por bairro Top 10")
fig.show()

### Salvar os dados na camada gold
zap_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(f"{path_gold}/zap_bairros")



logging.info("bairros gravado com sucesso na camada gold!")


BAIRRO,PRECO_MEDIO
COMITECO,21000.0
GARÇAS,11500.0
ANCHIETA,8571.43
SAVASSI,7335.29
LOURDES,7082.0
BELVEDERE,7013.16
SANTO AGOSTINHO,6700.0
BOA VIAGEM,6600.0
CAMPO ALEGRE,6500.0
FUNCIONÁRIOS,6262.73


In [0]:
## Rank bairros mais baratos

zap_bairros_baratos = zap_gold.filter(col("PRECO")<=1000000).groupBy("BAIRRO") \
    .agg(mean("PRECO").alias("PRECO_MEDIO"))\
    .withColumn("PRECO_MEDIO", round(col("PRECO_MEDIO"), 2))\
    .orderBy(col("PRECO_MEDIO").asc())\
    .limit(10)
    

display(zap_bairros_baratos)

zap_gold.write.format("delta").mode("append").option("mergeSchema", "true").save(f"{path_gold}/zap_bairros_baratos")
logging.info("bairros baratos gravado com sucesso na camada gold!")

zap_bairros_baratos.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.zap_imoveis.zap_bairros_baratos")



BAIRRO,PRECO_MEDIO
JAQUELINE,1200.0
CARDOSO (BARREIRO),1550.0
LAGOINHA LEBLON (VENDA NOVA),1590.0
VENDA NOVA,1750.0
TEIXEIRA DIAS (BARREIRO),1800.0
SERRA VERDE (VENDA NOVA),1800.0
POMPÉIA,1900.0
MANTIQUEIRA,1900.0
CONJUNTO PAULO VI,1950.0
PONGELUPE (BARREIRO),1950.0


###Analises de preço e custo :
Diferença entre o custo total por segmento e região 

In [0]:
## Diferença entre o custo total por segmento e região 

zap_custo_total = zap_gold.filter(col("PRECO")<=1000000).groupBy("REGIÃO","SEGMENTO") \
    .agg(mean("CUSTO_TOTAL").alias("CUSTO_TOTAL_MEDIO"))\
    .withColumn("CUSTO_TOTAL_MEDIO", round(col("CUSTO_TOTAL_MEDIO"), 2))\
    .orderBy(col("CUSTO_TOTAL_MEDIO").desc())
    

display(zap_custo_total)

## Plotar um grafico de colunas 

fig =px.bar (zap_custo_total, x="REGIÃO", y="CUSTO_TOTAL_MEDIO", color="SEGMENTO", title="Custo total por região e segmento", barmode="group")

fig.show()





### Salvar os dados na camada gold
zap_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(f"{path_gold}/zap_custo_total")

logging.info("custo total gravado com sucesso na camada gold!")

REGIÃO,SEGMENTO,CUSTO_TOTAL_MEDIO
OESTE,COMERCIAL,11168.75
BARREIRO,RESIDENCIAL,6562.68
CENTRO-SUL,COMERCIAL,6308.41
CENTRO-SUL,RESIDENCIAL,6047.21
NORDESTE,COMERCIAL,5550.0
OESTE,RESIDENCIAL,4654.39
NORTE,RESIDENCIAL,4644.77
BARREIRO,COMERCIAL,4411.5
NOROESTE,COMERCIAL,4392.5
NOROESTE,RESIDENCIAL,4356.46


###Análises de mercado :
Quantidade de imóveis por bairro

In [0]:
## Quantidade de imoveis por regiao

zap_quantidade = zap_gold.groupBy("REGIÃO") \
    .agg(count("REGIÃO").alias("QUANTIDADE_IMOVEIS"))\
    .orderBy(col("QUANTIDADE_IMOVEIS").desc()).limit(20)
    

display(zap_quantidade)

## Plotar um headmap
fig= px.funnel(zap_quantidade, x="REGIÃO", y="QUANTIDADE_IMOVEIS", orientation="v", color="REGIÃO", title="Quantidade de imoveis por região")
fig.show()

### Salvar os dados na camada gold
zap_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(f"{path_gold}/zap_quantidade")

logging.info("quantidade gravado com sucesso na camada gold!")

REGIÃO,QUANTIDADE_IMOVEIS
CENTRO-SUL,381
BARREIRO,276
OESTE,131
NORDESTE,100
NORTE,71
OUTROS,54
NOROESTE,37


###Análises de mercado :
Distribuição por tipo de imóvel

In [0]:
## Distribuição por tipo de imovel

zap_tipo = zap_gold.groupBy("AREA", "TIPO_DE_IMOVEL", "REGIÃO") \
    .agg(count("AREA").alias("QUANTIDADE_IMOVEIS"))\
    .orderBy(col("QUANTIDADE_IMOVEIS").desc())
    
display(zap_tipo)

## grafico linha

fig = px.bar(zap_tipo, x="REGIÃO", y="QUANTIDADE_IMOVEIS", color="TIPO_DE_IMOVEL", title="Quantidade de imoveis por região e tipo")
fig.show()

### Salvar os dados na camada gold
zap_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(f"{path_gold}/zap_tipo")

logging.info("tipo gravado com sucesso na camada gold!")






AREA,TIPO_DE_IMOVEL,REGIÃO,QUANTIDADE_IMOVEIS
100,APARTAMENTO,CENTRO-SUL,53
85,APARTAMENTO,BARREIRO,23
90,APARTAMENTO,CENTRO-SUL,22
80,APARTAMENTO,CENTRO-SUL,19
100,APARTAMENTO,BARREIRO,16
90,APARTAMENTO,BARREIRO,15
120,APARTAMENTO,CENTRO-SUL,14
80,APARTAMENTO,BARREIRO,14
110,APARTAMENTO,BARREIRO,12
65,APARTAMENTO,BARREIRO,11


In [0]:
## Salvar as tabelas criadas na gold

gold_metricas.write.format("delta").mode("overwrite").saveAsTable("workspace.zap_imoveis.gold_metricas")
logging.info("metricas gravado com sucesso na camada gold!")

zap_gold.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("workspace.zap_imoveis.zap_gold")
logging.info("zap_gold gravado com sucesso na camada gold!")

zap_media_regiao.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("workspace.zap_imoveis.zap_media_regiao")
logging.info("media por região gravado com sucesso na camada gold!")

zap_media_tipo.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("workspace.zap_imoveis.zap_media_tipo")
logging.info("media por tipo gravado com sucesso na camada gold!")


zap_bairros.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.zap_imoveis.zap_bairros")
logging.info("bairros gravado com sucesso na camada gold!")

zap_bairros.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.zap_imoveis.zap_bairros_baratos")
logging.info("bairros baratos gravado com sucesso na camada gold!")

zap_custo_total.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.zap_imoveis.zap_custo_total")
logging.info("custo total gravado com sucesso na camada gold!")

zap_tipo.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.zap_imoveis.zap_tipo")
logging.info("tipo gravado com sucesso na camada gold!")




In [0]:
display(zap_gold)

#verificar total de linhas
zap_gold.count()

PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,BANHEIROS,VAGAS,IPTU,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO,CUSTO_TOTAL,DATA_CARGA,ID
4500000,0,100,3,SAVASSI,3,3.0,1500,CASA,RESIDENCIAL,CENTRO-SUL,45000.0,4501500,2026-05-17T15:40:55.088Z,0
335000,330,100,3,BURITIS,2,1.0,145,APARTAMENTO,RESIDENCIAL,BARREIRO,3350.0,335475,2026-05-17T15:40:55.088Z,1
21000,0,332,3,COMITECO,6,3.0,7764,CASA,RESIDENCIAL,NOROESTE,63.25,28764,2026-05-17T15:40:55.088Z,2
25000,2200,264,3,LOURDES,3,3.0,1259,COBERTURA,RESIDENCIAL,CENTRO-SUL,94.7,28459,2026-05-17T15:40:55.088Z,3
25000,2300,105,3,SAVASSI,3,0.0,764,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,238.1,28064,2026-05-17T15:40:55.088Z,4
18000,4074,304,3,BELVEDERE,2,4.0,1334,COBERTURA,RESIDENCIAL,CENTRO-SUL,59.21,23408,2026-05-17T15:40:55.088Z,5
14000,0,256,3,BELVEDERE,4,4.0,7206,CASA,RESIDENCIAL,CENTRO-SUL,54.69,21206,2026-05-17T15:40:55.088Z,6
16500,2000,150,3,ANCHIETA,4,4.0,850,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,19350,2026-05-17T15:40:55.088Z,7
16500,2000,150,3,ANCHIETA,4,4.0,219,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,18719,2026-05-17T15:40:55.088Z,8
11000,0,365,3,SÃO PEDRO,2,3.0,6038,IMOVEL COMERCIAL,COMERCIAL,CENTRO-SUL,30.14,17038,2026-05-17T15:40:55.088Z,9


1050